# DCASE 2025 Task 3 — SELD Visualization
Visualize predicted vs. ground-truth sound events for the dev-test set.

In [ ]:
import os, glob
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
%matplotlib inline
matplotlib.rcParams['figure.dpi'] = 120

## Configuration
Update `PRED_DIR` to point to your prediction output folder.

In [ ]:
import os, glob

# Auto-detect latest output dir, or set manually
output_dirs = sorted(glob.glob('outputs/*/dev-test'))
PRED_DIR = output_dirs[-1] if output_dirs else 'outputs/SELDnet_audio_multiACCDOA_20250331_152343/dev-test'

GT_DIR     = '../DCASE2025_SELD_dataset/metadata_dev'
NB_CLASSES = 13
NB_EXAMPLES = 6

CLASS_NAMES = [
    'Female speech', 'Male speech', 'Clapping', 'Telephone', 'Laughter',
    'Domestic sounds', 'Footsteps', 'Door', 'Music',
    'Musical instrument', 'Water tap', 'Bell', 'Knock'
]

COLORS = plt.cm.tab20.colors

print(f'Using predictions from: {PRED_DIR}')

## Helper Functions

In [ ]:
def read_csv(path):
    """Return dict: frame -> list of [class, azimuth, distance_m]"""
    data = {}
    if not os.path.exists(path):
        return data
    with open(path) as f:
        lines = f.readlines()[1:]  # skip header
    for line in lines:
        parts = line.strip().split(',')
        if len(parts) < 5:
            continue
        frame = int(parts[0])
        cls   = int(parts[1])
        az    = float(parts[3])
        dist  = float(parts[4]) / 100.0  # cm -> m
        data.setdefault(frame, []).append([cls, az, dist])
    return data


def data_to_arrays(data, nb_frames=50):
    active = np.zeros((nb_frames, NB_CLASSES), dtype=bool)
    az     = np.full((nb_frames, NB_CLASSES), np.nan)
    dist   = np.full((nb_frames, NB_CLASSES), np.nan)
    for frame, events in data.items():
        if frame >= nb_frames:
            continue
        for cls, a, d in events:
            active[frame, cls] = True
            az[frame, cls]     = a
            dist[frame, cls]   = d
    return active, az, dist


# Build GT lookup once (filename -> full path) to avoid per-file recursive glob
print('Building GT file lookup...')
gt_lookup = {}
for path in glob.glob(os.path.join(GT_DIR, '**', '*.csv'), recursive=True):
    gt_lookup[os.path.basename(path)] = path
print(f'  {len(gt_lookup)} GT files indexed')

pred_files = sorted(glob.glob(os.path.join(PRED_DIR, '*.csv')))
print(f'Found {len(pred_files)} prediction files')

## Plot 1: Class-wise Activity — GT vs Predicted

In [ ]:
pred_counts = np.zeros(NB_CLASSES)
gt_counts   = np.zeros(NB_CLASSES)

for pf in pred_files:
    gf = gt_lookup.get(os.path.basename(pf))
    for events in read_csv(pf).values():
        for cls, _, _ in events:
            pred_counts[cls] += 1
    if gf:
        for events in read_csv(gf).values():
            for cls, az, dist in events:
                gt_counts[cls] += 1

x = np.arange(NB_CLASSES)
w = 0.38
fig, ax = plt.subplots(figsize=(13, 4))
ax.bar(x - w/2, gt_counts,   w, label='Ground Truth', color='steelblue', alpha=0.85)
ax.bar(x + w/2, pred_counts, w, label='Predicted',    color='coral',     alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES, rotation=35, ha='right', fontsize=9)
ax.set_ylabel('Active frame count')
ax.set_title('Class-wise Activity: Ground Truth vs Predicted (dev-test)')
ax.legend()
plt.tight_layout()
os.makedirs('figures', exist_ok=True)
plt.savefig('figures/activity_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## Plot 2: Azimuth Error Distribution per Class

In [ ]:
errors_per_class = {c: [] for c in range(NB_CLASSES)}

for pf in pred_files:
    gf = gt_lookup.get(os.path.basename(pf))
    if not gf:
        continue
    pred = read_csv(pf)
    gt   = read_csv(gf)
    for frame in set(pred.keys()) & set(gt.keys()):
        gt_map   = {e[0]: e[1] for e in gt[frame]}
        pred_map = {e[0]: e[1] for e in pred[frame]}
        for cls in set(gt_map) & set(pred_map):
            err = abs(gt_map[cls] - pred_map[cls])
            errors_per_class[cls].append(min(err, 360 - err))

fig, ax = plt.subplots(figsize=(13, 4))
bp = ax.boxplot(
    [errors_per_class[c] if errors_per_class[c] else [0] for c in range(NB_CLASSES)],
    patch_artist=True, notch=False, showfliers=False
)
for patch, color in zip(bp['boxes'], COLORS[:NB_CLASSES]):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)
ax.set_xticks(range(1, NB_CLASSES + 1))
ax.set_xticklabels(CLASS_NAMES, rotation=35, ha='right', fontsize=9)
ax.set_ylabel('Azimuth error (°)')
ax.set_title('Azimuth Error Distribution per Class (matched frames, dev-test)')
ax.axhline(20, color='red', linestyle='--', linewidth=1, label='20° threshold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('figures/az_error_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## Plot 3: Example Clip Timelines (Activity + Azimuth)
Shows predicted vs. ground truth for individual 5-second clips.

In [ ]:
os.makedirs('figures', exist_ok=True)

# Pick NB_EXAMPLES clips that have ground truth activity
examples = []
for pf in pred_files:
    gf = gt_lookup.get(os.path.basename(pf))
    if gf and os.path.getsize(gf) > 100:
        examples.append((pf, gf))
    if len(examples) >= NB_EXAMPLES:
        break

print(f'Plotting {len(examples)} example clips...')

for i, (pf, gf) in enumerate(examples):
    pred = read_csv(pf)
    gt   = read_csv(gf)
    nb_frames  = 50
    time_axis  = np.arange(nb_frames) * 0.1

    pred_active, pred_az, _ = data_to_arrays(pred, nb_frames)
    gt_active,   gt_az,   _ = data_to_arrays(gt,   nb_frames)

    active_classes = np.where(gt_active.any(0) | pred_active.any(0))[0]
    if len(active_classes) == 0:
        continue

    fig, axes = plt.subplots(len(active_classes), 2,
                              figsize=(12, 2.2 * len(active_classes)),
                              sharex=True)
    if len(active_classes) == 1:
        axes = axes[np.newaxis, :]

    title = os.path.basename(pf).replace('.csv', '')
    fig.suptitle(title, fontsize=9)

    for row, cls in enumerate(active_classes):
        color = COLORS[cls % len(COLORS)]

        # Activity
        ax_act = axes[row, 0]
        ax_act.fill_between(time_axis, 0, gt_active[:, cls].astype(float),
                            alpha=0.35, color=color)
        ax_act.step(time_axis, pred_active[:, cls].astype(float),
                    where='mid', color=color, linewidth=1.5)
        ax_act.set_ylim(-0.1, 1.5)
        ax_act.set_yticks([])
        ax_act.set_ylabel(CLASS_NAMES[cls], fontsize=8, rotation=0,
                           ha='right', va='center', labelpad=5)
        if row == 0:
            ax_act.set_title('Activity  (fill=GT, line=Pred)', fontsize=8)

        # Azimuth
        ax_az = axes[row, 1]
        ax_az.plot(time_axis, np.where(gt_active[:, cls],   gt_az[:, cls],   np.nan),
                   color=color, alpha=0.5, linewidth=2.5, linestyle='--', label='GT')
        ax_az.plot(time_axis, np.where(pred_active[:, cls], pred_az[:, cls], np.nan),
                   color=color, linewidth=1.5, label='Pred')
        ax_az.set_ylim(-185, 185)
        ax_az.set_yticks([-180, -90, 0, 90, 180])
        ax_az.tick_params(labelsize=7)
        if row == 0:
            ax_az.set_title('Azimuth (°)', fontsize=8)
            ax_az.legend(fontsize=7, loc='upper right')

    axes[-1, 0].set_xlabel('Time (s)', fontsize=8)
    axes[-1, 1].set_xlabel('Time (s)', fontsize=8)
    plt.tight_layout()
    out_path = f'figures/clip_timeline_{i+1:02d}.png'
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'  Saved {out_path}')

## Plot 4: Distance Error Distribution per Class

In [ ]:
dist_errors_per_class = {c: [] for c in range(NB_CLASSES)}

for pf in pred_files:
    gf = gt_lookup.get(os.path.basename(pf))
    if not gf:
        continue
    pred = read_csv(pf)
    gt   = read_csv(gf)
    for frame in set(pred.keys()) & set(gt.keys()):
        gt_map   = {e[0]: e[2] for e in gt[frame]}   # cls -> dist_m
        pred_map = {e[0]: e[2] for e in pred[frame]}
        for cls in set(gt_map) & set(pred_map):
            dist_errors_per_class[cls].append(abs(gt_map[cls] - pred_map[cls]))

fig, ax = plt.subplots(figsize=(13, 4))
bp = ax.boxplot(
    [dist_errors_per_class[c] if dist_errors_per_class[c] else [0] for c in range(NB_CLASSES)],
    patch_artist=True, notch=False, showfliers=False
)
for patch, color in zip(bp['boxes'], COLORS[:NB_CLASSES]):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)
ax.set_xticks(range(1, NB_CLASSES + 1))
ax.set_xticklabels(CLASS_NAMES, rotation=35, ha='right', fontsize=9)
ax.set_ylabel('Distance error (m)')
ax.set_title('Distance Error Distribution per Class (matched frames, dev-test)')
plt.tight_layout()
plt.savefig('figures/dist_error_distribution.png', dpi=150, bbox_inches='tight')
plt.show()